# ISIC 2017 — Full 600-Image Evaluation Pipeline

Runs the complete four-scheme FAE ranking pipeline on the full ISIC 2017
test set (600 images × 2 models × 7 FAE methods × 11 active metrics).

| Step | Cell | Estimated runtime (T4) |
|---|---|---|
| Vertical slice (600 images) | 6 | **90–120 min** |
| Meta-evaluation (NR + AR, 600 images) | 7 | **4–5 h** |
| Four-scheme comparison | 9 | < 1 min |
| Wilcoxon test | 10 | < 1 s |
| **Total** | | **~5–7 h** |

**Recommendation**: Use **Colab Pro** (12-hour sessions). Cell 7
(meta-evaluation) writes a checkpoint after every (model, FAE) pair;
a disconnect mid-run does **not** lose progress — re-run cell 7 to resume.

**Prerequisites**
- Runtime: **T4 GPU** — Runtime → Change runtime type → T4 GPU
- Google Drive must contain:
  - `thesis/weights/resnet18_isic2017.pth`
  - `thesis/weights/squeezenet_isic2017.pth`
  - `thesis/data/images/test/` and `thesis/data/masks/test/` (600 images + masks)
    *(saved by `colab_vertical_slice.ipynb` cell 9; falls back to ISIC download)*

**Output CSVs**
- `results/vertical_slice_7fae_12metrics_FULL.csv` — 92,400 rows
- `results/meta_evaluation_reliability_FULL.csv`
- `results/ranking_comparison_FULL.csv`

## 1. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DRIVE_THESIS = '/content/drive/MyDrive/thesis'

import os
os.makedirs(f'{DRIVE_THESIS}/results', exist_ok=True)
print(f'Drive mounted. Thesis folder: {DRIVE_THESIS}')

## 2. Clone Repository

In [ ]:
import os

REPO_DIR = '/content/fae-metrics-master-thesis'

if os.path.exists(REPO_DIR):
    %cd {REPO_DIR}
    !git pull
else:
    !git clone https://github.com/dawkopagh/fae-metrics-master-thesis.git {REPO_DIR}
    %cd {REPO_DIR}

print(f'Working directory: {os.getcwd()}')
print('HEAD commit:', end=' ')
!git rev-parse HEAD

## 3. Install Dependencies

Pins `quantus==0.6.0` explicitly until `requirements.txt` carries version pins.

In [ ]:
!pip install -q -r requirements.txt
!pip install -q quantus==0.6.0

import captum, quantus, torch, scipy
print(f'captum  {captum.__version__}')
print(f'quantus {quantus.__version__}')
print(f'torch   {torch.__version__}')
print(f'scipy   {scipy.__version__}')
print(f'CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')

## 4. Copy Weights from Drive

SHA-256 values are from `weights/README.md` (trained 2026-04-22, full ISIC 2017).

In [ ]:
DRIVE_THESIS   = '/content/drive/MyDrive/thesis'
RESNET_SHA     = '593dcb844b8359550e3d84667475bbd845f45a6cb388356480c0a55cb5173430'
SQUEEZENET_SHA = '8bbb43bbba4ee81e58e354295420bea33e55cfa2be11c31d82dce272d03b092d'

import os
os.makedirs('weights', exist_ok=True)

!cp {DRIVE_THESIS}/weights/resnet18_isic2017.pth   weights/resnet18_isic2017.pth
!cp {DRIVE_THESIS}/weights/squeezenet_isic2017.pth weights/squeezenet_isic2017.pth

print('SHA-256 checksums (actual):')
!sha256sum weights/resnet18_isic2017.pth weights/squeezenet_isic2017.pth
print(f'\nExpected resnet18:   {RESNET_SHA}')
print(f'Expected squeezenet: {SQUEEZENET_SHA}')

## 5. Copy Full Test Data from Drive

Copies 600 test images + 600 segmentation masks (~800 MB) from Drive.
These were saved by `colab_vertical_slice.ipynb` cell 9.
Falls back to a full ISIC 2017 download (~6 GB, ~20 min) if not found.

In [ ]:
DRIVE_THESIS = '/content/drive/MyDrive/thesis'

import os, sys
sys.path.insert(0, '.')

# Check whether both test-split directories exist on Drive.
_drive_has_images = os.path.exists(f'{DRIVE_THESIS}/data/images/test')
_drive_has_masks  = os.path.exists(f'{DRIVE_THESIS}/data/masks/test')

if _drive_has_images and _drive_has_masks:
    for split_dir in ['data/images/test', 'data/masks/test']:
        src = f'{DRIVE_THESIS}/{split_dir}'
        os.makedirs(split_dir, exist_ok=True)
        !cp -r {src}/* {split_dir}/
        n = sum(len(files) for _, _, files in os.walk(split_dir))
        print(f'{split_dir}: {n} files copied from Drive')
else:
    print('Test data not on Drive — downloading from ISIC 2017 (~6 GB, ~20 min).')
    from src.data.download_isic import download_isic2017
    download_isic2017(dest_dir='data', skip_existing=True)

# Verify: expect exactly 600 test images across 3 classes.
_test_classes = ['melanoma', 'nevus', 'seborrheic_keratosis']
n_images = sum(
    len([f for f in os.listdir(f'data/images/test/{cls}') if not f.startswith('.')])
    for cls in _test_classes
    if os.path.exists(f'data/images/test/{cls}')
)
print(f'\nTest images found: {n_images}')
assert n_images == 600, f'Expected 600 test images, got {n_images}'
print('Data scaffold ready.')

## 6. Vertical Slice — 600 Images (~90–120 min)

Runs `experiments/vertical_slice.py` over the full test split:
2 models × 7 FAE methods × 600 images × 11 active metrics = **92,400 rows**.

`non_sensitivity` is excluded (all-NaN by design) to save 8,400 useless rows;
the filename retains `12metrics` for backwards compatibility with downstream code.

Attribution maps are cached to `attributions_cache/`; re-runs skip attribution
and only recompute metric scores.

In [ ]:
import os, time, pandas as pd
os.makedirs('results', exist_ok=True)

SLICE_FULL = 'results/vertical_slice_7fae_12metrics_FULL.csv'
_TMP_RAW   = '/tmp/vertical_slice_raw_full.csv'

print('Vertical slice: 2 models × 7 FAE × 600 images × up to 12 metrics')
print('Estimated runtime: 90–120 min on T4')
print('Attribution cache speeds subsequent re-runs significantly.')
print()

t0 = time.time()
!python experiments/vertical_slice.py --output-csv {_TMP_RAW}
elapsed = time.time() - t0

# Filter out non_sensitivity rows (all-NaN; excluded to keep row count clean).
df_raw  = pd.read_csv(_TMP_RAW)
df_full = df_raw[df_raw['metric'] != 'non_sensitivity'].reset_index(drop=True)
df_full.to_csv(SLICE_FULL, index=False)

# 11 active metrics × 600 images × 2 models × 7 FAE = 92,400
EXPECTED_ROWS = 92_400
print(f'\nRaw rows (incl. non_sensitivity if present): {len(df_raw)}')
print(f'After filter: {len(df_full)} rows  (expected {EXPECTED_ROWS})')
assert len(df_full) == EXPECTED_ROWS, (
    f'Expected {EXPECTED_ROWS} rows after filtering non_sensitivity, '
    f'got {len(df_full)}. Check that all 7 FAE × 11 metrics × 600 images × '
    f'2 models completed without unexpected NaN patterns.'
)
print(f'Elapsed: {elapsed / 60:.1f} min')

print('\nNaN counts per metric (non-zero only):')
nan_counts = df_full.groupby('metric')['score'].apply(lambda s: s.isna().sum())
nan_nonzero = nan_counts[nan_counts > 0]
print(nan_nonzero.to_string() if not nan_nonzero.empty else '  (none)')

## 7. Meta-Evaluation — 600 Images (~4–5 h)

Runs Noise Resilience (NR) + Adversarial Reactivity (AR) tests
(Hedström et al., TMLR 2024) for every eligible `(model, fae, metric)` triple.

**Runtime**: 4–5 hours on T4. Colab Pro (12-hour sessions) strongly recommended.
The function writes a checkpoint row after each `(model, FAE)` pair completes,
so a session disconnect does **not** lose progress.

**Resume**: re-running this cell skips any triple already written with
`status='completed'` or `status='skipped_nan'` in the output CSV.
You may safely disconnect and resume across multiple free-tier sessions.

**Decision point before starting**:
- This cell prints estimated total runtime before beginning.
- If the estimate exceeds your remaining session time, disconnect now
  and return later — checkpointing means no work is lost.

In [ ]:
import sys, os, time, datetime
import numpy as np, torch, quantus, pandas as pd
sys.path.insert(0, '.')
os.makedirs('results', exist_ok=True)

SLICE_FULL   = 'results/vertical_slice_7fae_12metrics_FULL.csv'
METAEVAL_OUT = 'results/meta_evaluation_reliability_FULL.csv'
PROGRESS_LOG = 'results/meta_eval_progress_FULL.log'

# ── Load the FULL slice CSV for pre-screening ──────────────────────────────
slice_df = pd.read_csv(SLICE_FULL)
print(f'Loaded slice: {len(slice_df)} rows')

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {device}')
if device == 'cpu':
    print('WARNING: No GPU detected. Meta-evaluation will be extremely slow on CPU.')

from src.models.classifiers  import load_resnet18, load_squeezenet
from src.data.isic_dataset   import ISIC2017Dataset
from src.pipeline            import _make_explain_func
from src.meta_evaluation.metaquantus_wrapper import (
    run_meta_evaluation_full, screen_meta_eval_candidates,
)

# ── Models ────────────────────────────────────────────────────────────────
models = {
    'resnet18':   load_resnet18('weights/resnet18_isic2017.pth',   device=device),
    'squeezenet': load_squeezenet('weights/squeezenet_isic2017.pth', device=device),
}

# ── Load all 600 test images and masks ────────────────────────────────────
dataset  = ISIC2017Dataset(root_dir='data', split='test', image_size=224, return_mask=True)
N_IMAGES = len(dataset)
print(f'Loading {N_IMAGES} test images and masks...')
samples  = [dataset[i] for i in range(N_IMAGES)]
images   = [s['image'] for s in samples]   # list of (3, 224, 224) tensors
masks    = [s['mask']  for s in samples]   # list of (1, 224, 224) arrays
print(f'Loaded: {len(images)} images, {len(masks)} masks')

# Batched target prediction from resnet18 (avoids 600 sequential GPU calls).
_BATCH = 64
_img_t = torch.stack([s['image'] for s in samples]).to(device)
targets = []
with torch.no_grad():
    for _i in range(0, N_IMAGES, _BATCH):
        _logits = models['resnet18'](_img_t[_i:_i + _BATCH])
        targets.extend(_logits.argmax(dim=1).tolist())
del _img_t  # free GPU memory before meta-eval
print(f'Targets computed for {len(targets)} images.')

# ── FAE explain functions (resnet18 arch; used as Quantus explain_func) ───
_FAE_NAMES = ['integrated_gradients', 'saliency', 'gradcam',
              'deep_lift', 'guided_backprop', 'lrp', 'occlusion']
fae_methods = {
    name: _make_explain_func(name, models['resnet18'], 'resnet18', device)
    for name in _FAE_NAMES
}

# ── Metric instances (same config as 12-image run) ─────────────────────────
metric_fns = {
    'faithfulness_correlation': quantus.FaithfulnessCorrelation(
        nr_runs=100, subset_size=224, perturb_baseline='black',
        normalise=True, abs=False, return_aggregate=False, disable_warnings=True),
    'pixel_flipping': quantus.PixelFlipping(
        features_in_step=224, perturb_baseline='black', normalise=True, abs=False,
        return_aggregate=False, return_auc_per_sample=True, disable_warnings=True),
    'max_sensitivity': quantus.MaxSensitivity(
        nr_samples=10, lower_bound=0.2, normalise=False, abs=False,
        return_aggregate=False, disable_warnings=True),
    'avg_sensitivity': quantus.AvgSensitivity(
        nr_samples=10, lower_bound=0.2, normalise=False, abs=False,
        return_aggregate=False, disable_warnings=True),
    'relevance_mass_accuracy': quantus.RelevanceMassAccuracy(
        normalise=True, abs=False, return_aggregate=False, disable_warnings=True),
    'pointing_game': quantus.PointingGame(
        normalise=True, abs=True, return_aggregate=False, disable_warnings=True),
    'sparseness': quantus.Sparseness(
        abs=True, normalise=True, return_aggregate=False, disable_warnings=True),
    'complexity': quantus.Complexity(
        abs=True, normalise=True, return_aggregate=False, disable_warnings=True),
    'model_parameter_randomisation': quantus.ModelParameterRandomisation(
        layer_order='top_down', normalise=True, abs=True,
        return_average_correlation=True, return_aggregate=False, disable_warnings=True),
    'random_logit': quantus.RandomLogit(
        num_classes=3, abs=True, normalise=True,
        return_aggregate=False, disable_warnings=True),
    'completeness': quantus.Completeness(
        abs=False, normalise=False, perturb_baseline='black',
        return_aggregate=False, disable_warnings=True),
}

# ── Pre-screening + runtime estimate ──────────────────────────────────────
candidates = screen_meta_eval_candidates(slice_df, min_valid_fraction=0.5)
eligible   = int(candidates['run_meta_eval'].sum())
skipped    = len(candidates) - eligible
print(f'\nEligible triples: {eligible}  |  Skipped (low coverage): {skipped}')

# Check how many are already done (resume case).
_done = 0
if os.path.exists(METAEVAL_OUT):
    _existing = pd.read_csv(METAEVAL_OUT)
    _done = int(_existing['status'].isin(['completed', 'skipped_nan']).sum())
    print(f'Resume: {_done} triples already done in {METAEVAL_OUT}')

# Reference: 12-image run took ~50 s/triple on T4.
# 600 images is 50× more data → scale estimate accordingly.
_remaining = max(0, eligible - _done)
_est_sec   = _remaining * 50 * 50  # 50x per triple * 50s baseline
print(f'Remaining triples: {_remaining}')
print(f'Estimated remaining runtime: {_est_sec / 3600:.1f} h  '
      f'({_est_sec / 60:.0f} min)')
print()
print('Colab Pro (12-hour sessions) is strongly recommended.')
print('Safe to disconnect: checkpoints write after each (model, FAE) pair.')
print('To resume later: re-run this cell — completed triples are skipped.')
print(f'\nStarting at: {datetime.datetime.now().isoformat(timespec="minutes")}')
print()

# ── Run ────────────────────────────────────────────────────────────────────
result_df = run_meta_evaluation_full(
    vertical_slice_df=slice_df,
    models=models,
    metric_fns=metric_fns,
    fae_methods=fae_methods,
    images=images,
    targets=targets,
    masks=masks,
    device=device,
    n_seeds=5,
    n_levels=5,
    output_csv=METAEVAL_OUT,
    progress_log=PROGRESS_LOG,
)

print(f'\nThis session: {len(result_df)} triples processed.')
if os.path.exists(METAEVAL_OUT):
    _all = pd.read_csv(METAEVAL_OUT)
    print('Overall CSV status:')
    print(_all['status'].value_counts().to_string())

## 8. Summarise Meta-Evaluation Results

Reads the checkpoint CSV and prints per-metric NR / AR / combined reliability.
Can be run at any point during cell 7 to check partial progress.

In [ ]:
import pandas as pd

METAEVAL_OUT = 'results/meta_evaluation_reliability_FULL.csv'

rel_df    = pd.read_csv(METAEVAL_OUT)
completed = rel_df[rel_df['status'] == 'completed']

print(f'Total rows in CSV:  {len(rel_df)}')
print(f'Completed:          {len(completed)}')
print(f'Skipped (NaN):      {(rel_df["status"]=="skipped_nan").sum()}')
print(f'Failed:             {(rel_df["status"]=="failed").sum()}')
print()

if len(completed) > 0:
    summary = (
        completed
        .groupby('metric')[['nr_score', 'ar_score', 'combined_reliability']]
        .mean()
        .round(3)
        .sort_values('combined_reliability', ascending=False)
    )
    print('=== Mean reliability per metric (completed triples only) ===')
    print(summary.to_string())

    print()
    print('=== Metrics with combined_reliability < 0.5 ===')
    low = summary[summary['combined_reliability'] < 0.5]
    print(low.to_string() if not low.empty else '  (none below 0.5)')
else:
    print('No completed triples yet — cell 7 may still be running.')

## 9. Four-Scheme Ranking Comparison

Runs `experiments/compare_rankings.py` with the FULL CSVs as inputs
and prints the three ranking tables (same format as the 12-image preliminary run).

Requires: cells 6 and 7 complete (or their output CSVs available on disk).

In [ ]:
SLICE_FULL   = 'results/vertical_slice_7fae_12metrics_FULL.csv'
METAEVAL_OUT = 'results/meta_evaluation_reliability_FULL.csv'
RANKING_OUT  = 'results/ranking_comparison_FULL.csv'

import os
for _path in [SLICE_FULL, METAEVAL_OUT]:
    assert os.path.exists(_path), f'Required input not found: {_path}'

!python experiments/compare_rankings.py \
    --slice-csv       {SLICE_FULL} \
    --reliability-csv {METAEVAL_OUT} \
    --output-csv      {RANKING_OUT}

import pandas as pd
df_rank = pd.read_csv(RANKING_OUT)
print(f'ranking_comparison_FULL.csv: {len(df_rank)} rows')
print(df_rank[['model', 'fae_method', 'image_id',
               'effectiveness_mqdiscount', 'effectiveness_single_fc']].head(5).to_string(index=False))

## 10. Wilcoxon Signed-Rank Test

Tests whether the mqdiscount and single_fc effectiveness scores are
systematically different across all (model, image, fae) triples.

This is the sole significance test retained for the thesis (two-sided,
default `alternative='two-sided'`). α = 0.05.

In [ ]:
import pandas as pd
from scipy.stats import wilcoxon

RANKING_OUT = 'results/ranking_comparison_FULL.csv'

df = pd.read_csv(RANKING_OUT)

# Drop rows where either scheme has NaN (wilcoxon requires complete pairs).
_valid = df[['effectiveness_mqdiscount', 'effectiveness_single_fc']].dropna()
print(f'Valid paired observations: {len(_valid)} of {len(df)}')

mq  = _valid['effectiveness_mqdiscount'].values
sfc = _valid['effectiveness_single_fc'].values

stat, p = wilcoxon(mq, sfc)

print()
print('=== Wilcoxon signed-rank test: mqdiscount vs single_fc ===')
print(f'  n (paired observations) : {len(mq)}')
print(f'  W statistic             : {stat:.1f}')
print(f'  p-value (two-sided)     : {p:.4g}')
print()
if p < 0.05:
    print('  Result: SIGNIFICANT (p < 0.05)')
    print('  mqdiscount and single_fc produce systematically different '
          'effectiveness scores.')
else:
    print('  Result: NOT significant (p >= 0.05)')
    print('  No systematic difference detected between mqdiscount and single_fc.')

## 11. Copy Full Results to Drive

Saves all three FULL output CSVs to Google Drive for local download.
Also saves the full test-split images and masks if they are not already on Drive
(so that future sessions can skip the ~20-min ISIC download).

In [ ]:
import shutil, os

DRIVE_THESIS = '/content/drive/MyDrive/thesis'

# ── Save FULL result CSVs ──────────────────────────────────────────────────
_full_csvs = [
    'results/vertical_slice_7fae_12metrics_FULL.csv',
    'results/meta_evaluation_reliability_FULL.csv',
    'results/ranking_comparison_FULL.csv',
    'results/meta_eval_progress_FULL.log',  # progress log for diagnostics
]

for src in _full_csvs:
    if os.path.exists(src):
        dest = f'{DRIVE_THESIS}/{src}'
        os.makedirs(os.path.dirname(dest), exist_ok=True)
        shutil.copy2(src, dest)
        _size_kb = os.path.getsize(dest) // 1024
        print(f'Saved  {src} → Drive  ({_size_kb} KB)')
    else:
        print(f'SKIP   {src}  (not found — cell may not have completed)')

# ── Save test-split images + masks to Drive for future sessions ────────────
for _data_dir in ['data/images/test', 'data/masks/test']:
    _dest = f'{DRIVE_THESIS}/{_data_dir}'
    if os.path.exists(_data_dir) and not os.path.exists(_dest):
        os.makedirs(_dest, exist_ok=True)
        !cp -r {_data_dir}/* {_dest}/
        _n = sum(len(files) for _, _, files in os.walk(_data_dir))
        print(f'Saved  {_data_dir}: {_n} files → Drive')
    elif os.path.exists(_dest):
        print(f'Skip   {_data_dir}: already on Drive')
    else:
        print(f'WARN   {_data_dir}: not found locally — nothing copied')

print('\nDone. Download results/ to local repo before the next session:')
print('  thesis/results/vertical_slice_7fae_12metrics_FULL.csv')
print('  thesis/results/meta_evaluation_reliability_FULL.csv')
print('  thesis/results/ranking_comparison_FULL.csv')